## 8-6 ##

In [2]:
import sys
!{sys.executable} -m ensurepip --upgrade

Looking in links: c:\Users\user\AppData\Local\Temp\tmp532ec21i
Processing c:\users\user\appdata\local\temp\tmp532ec21i\pip-23.0.1-py3-none-any.whl


In [3]:
%pip install opencv-python numpy tensorflow matplotlib PyQt5

     ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
     ---------------------------------------- 0.5/44.0 MB 10.5 MB/s eta 0:00:05
     -- ------------------------------------- 2.7/44.0 MB 28.5 MB/s eta 0:00:02
     --- ------------------------------------ 4.1/44.0 MB 29.3 MB/s eta 0:00:02
     ----- ---------------------------------- 5.7/44.0 MB 30.2 MB/s eta 0:00:02
     -------- ------------------------------- 9.2/44.0 MB 36.7 MB/s eta 0:00:01
     ---------- ---------------------------- 11.5/44.0 MB 43.5 MB/s eta 0:00:01
     ------------ -------------------------- 13.9/44.0 MB 46.7 MB/s eta 0:00:01
     ------------ -------------------------- 14.4/44.0 MB 50.4 MB/s eta 0:00:01
     ------------ -------------------------- 14.5/44.0 MB 40.9 MB/s eta 0:00:01
     ------------- ------------------------- 15.3/44.0 MB 34.4 MB/s eta 0:00:01
     -------------- ------------------------ 16.3/44.0 MB 28.5 MB/s eta 0:00:01
     --------------- ----------------------- 17.

In [6]:
import cv2 as cv
import numpy as np
from tensorflow.keras.applications.resnet50 import ResNet50,preprocess_input,decode_predictions

model=ResNet50(weights='imagenet')

img=cv.imread('rabbit.jpg')
x=np.reshape(cv.resize(img,(224,224)),(1,224,224,3))
x=preprocess_input(x)

preds=model.predict(x)
top5=decode_predictions(preds,top=5)[0]
print('예측 결과:',top5)

for i in range(5):
    cv.putText(img,top5[i][1]+':'+str(top5[i][2]),(10,20+i*20),cv.FONT_HERSHEY_SIMPLEX,0.5,(255,255,255),1)

cv.imshow('Recognition result',img)

cv.waitKey()
cv.destroyAllWindows()

35363/35363 [==============================] - 0s 0us/step
예측 결과: [('n02325366', 'wood_rabbit', 0.7264928), ('n02326432', 'hare', 0.2571298), ('n02328150', 'Angora', 0.0073944447), ('n01877812', 'wallaby', 0.0031294078), ('n01883070', 'wombat', 0.0010740497)]


## 8-7 ##

In [1]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten,Dense,Dropout,Rescaling
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications.densenet import DenseNet121
from tensorflow.keras.utils import image_dataset_from_directory
import pathlib

data_path=pathlib.Path('datasets/stanford_dogs/images/images')  # 압축 푼 실제 경로로 맞출 것

train_ds=image_dataset_from_directory(data_path,validation_split=0.2,subset='training',seed=123,image_size=(224,224),batch_size=16)
test_ds=image_dataset_from_directory(data_path,validation_split=0.2,subset='validation',seed=123,image_size=(224,224),batch_size=16)

base_model=DenseNet121(weights='imagenet',include_top=False,input_shape=(224,224,3))
cnn=Sequential()
cnn.add(Rescaling(1.0/255.0))
cnn.add(base_model)
cnn.add(Flatten())
cnn.add(Dense(1024,activation='relu'))
cnn.add(Dropout(0.75))
cnn.add(Dense(units=120,activation='softmax'))

cnn.compile(loss='sparse_categorical_crossentropy',optimizer=Adam(learning_rate=0.000001),metrics=['accuracy'])
hist=cnn.fit(train_ds,epochs=200,validation_data=test_ds,verbose=2)  # 연습용이면 10~20으로 줄이기

print('정확률=',cnn.evaluate(test_ds,verbose=0)[1]*100)

cnn.save('cnn_for_stanford_dogs.h5')   # 미세 조정된 모델을 파일에 저장

import pickle
f=open('dog_species_names.txt','wb')
pickle.dump(train_ds.class_names,f)
f.close()

import matplotlib.pyplot as plt

plt.plot(hist.history['accuracy'])
plt.plot(hist.history['val_accuracy'])
plt.title('Accuracy graph')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train','Validation'])
plt.grid()
plt.show()

plt.plot(hist.history['loss'])
plt.plot(hist.history['val_loss'])
plt.title('Loss graph')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train','Validation'])
plt.grid()
plt.show()

Found 20580 files belonging to 120 classes.
Using 16464 files for training.
Found 20580 files belonging to 120 classes.
Using 4116 files for validation.
29084464/29084464 [==============================] - 2s 0us/step
Epoch 1/200


KeyboardInterrupt: 

## 8-8 ##

In [ ]:
import cv2 as cv
import numpy as np
import tensorflow as tf
import winsound   # 윈도우 전용 (맥/리눅스면 삭제)
import pickle
import sys
from PyQt5.QtWidgets import *

cnn=tf.keras.models.load_model('cnn_for_stanford_dogs.h5')         # 모델 읽기
dog_species=pickle.load(open('dog_species_names.txt','rb'))        # 견종 이름

class DogSpeciesRecognition(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle('견종 인식')
        self.setGeometry(200,200,700,100)

        fileButton=QPushButton('강아지 사진 열기',self)
        recognitionButton=QPushButton('품종 인식',self)
        quitButton=QPushButton('나가기',self)

        fileButton.setGeometry(10,10,100,30)
        recognitionButton.setGeometry(110,10,100,30)
        quitButton.setGeometry(510,10,100,30)

        fileButton.clicked.connect(self.pictureOpenFunction)
        recognitionButton.clicked.connect(self.recognitionFunction)
        quitButton.clicked.connect(self.quitFunction)

    def pictureOpenFunction(self):
        fname=QFileDialog.getOpenFileName(self,'강아지 사진 읽기','./')
        self.img=cv.imread(fname[0])
        if self.img is None: sys.exit('파일을 찾을 수 없습니다.')

        cv.imshow('Dog image',self.img)

    def recognitionFunction(self):
        x=np.reshape(cv.resize(self.img,(224,224)),(1,224,224,3))
        # 정확도 개선하려면 위 줄 대신:
        # x=np.reshape(cv.resize(cv.cvtColor(self.img,cv.COLOR_BGR2RGB),(224,224)),(1,224,224,3))
        res=cnn.predict(x)[0]   # 예측
        top5=np.argsort(-res)[:5]
        top5_dog_species_names=[dog_species[i] for i in top5]
        for i in range(5):
            prob='('+str(res[top5[i]])+')'
            name=str(top5_dog_species_names[i]).split('-')[1]
            cv.putText(self.img,prob+name,(10,100+i*30),cv.FONT_HERSHEY_SIMPLEX,0.7,(255,255,255),2)
        cv.imshow('Dog image',self.img)
        winsound.Beep(1000,500)

    def quitFunction(self):
        cv.destroyAllWindows()
        self.close()

app=QApplication(sys.argv)
win=DogSpeciesRecognition()
win.show()
app.exec_()